# Тематическое моделирование корпуса כמיהה

Корпус: 20 752 употребления существительного כמיהה (файл `all_noun_forms.csv`). Метод: TF-IDF + NMF (неотрицательная матричная факторизация), число тем выбрано по когерентности NPMI. Контекст = окно из ~43 слов вокруг слова (`left` + `right`, без самой формы כמיהה).

In [ ]:
import pandas as pd, numpy as np, re, collections, itertools
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
CSV = 'all_noun_forms.csv'   # загрузите файл корпуса в Colab

## 1. Очистка
Удаление почти-дубликатов (одинаковое окно ±6 слов), огласовок, стоп-слов, всех форм כמיהה. Префиксы ו ה ב ל מ ש כ отсекаются, только если слово без префикса встречается в корпусе не реже (иначе «לב» превратился бы в «ב»).

In [ ]:
df=pd.read_csv(CSV,encoding='utf-8-sig')
df['ctx']=(df.left.fillna('')+' '+df.right.fillna('')).str.replace(r'</?s>',' ',regex=True)
# near-duplicate removal: same 12-word window around the keyword
key=(df.left.fillna('').str.split().str[-6:].str.join(' ')+'|'+df.right.fillna('').str.split().str[:6].str.join(' '))
df=df[~key.duplicated()].copy()
STOP=set("""של את על עם זה זו זאת הוא היא הם הן אני אנחנו אתה את אתם לא כן גם או אבל כי אם כל יש אין היה היתה היו יהיה להיות
מה מי איך למה כמו רק עוד כבר מאוד יותר פחות אחד אחת שני שתי בין אל עד אחרי לפני תחת מתוך בתוך בלי ללא כך כאן שם אז עכשיו
אותו אותה אותם שלו שלה שלהם שלי שלנו שלך לו לה להם לי לנו לך בו בה בהם בי בנו ממנו ממנה מהם אשר כאשר כדי בגלל לכן אולם אך
הזה הזאת האלה אלה אלו כאילו בעצם אפילו עצמו עצמה עצמם כלל כמה הרבה מעט די רבה רבים רבות מן כפי אף שוב תמיד לעולם הייתה
דרך פי וכו וגם וכן ולא ואת וזה שהוא שהיא שלא שזה שאין שיש כשהוא היום שנים שנה זמן ו ה ב ל מ ש כ את' יותר' כמיהה
דבר דברים אולי משהו פעם פעמים רוצה רוצים רצה בכל טוב טובה אותי אותך אותנו אותם רגע יכול יכולה יכולים אפשר הזו הזה יום ימים הכל
קשר בלתי אלא ידי וה באופן אינה אינו אינם חלק אנו מכן אחר אחרת אחרים שבו שבה זהו ועל בשנת המשך הרי עדיין ממש לגבי אצל כיצד
ככל כמובן שוב בכך בזה לכך מכל נראה נראית צריך צריכה צריכים לעשות עושה עושים אמר אומר אמרה אומרת כאן לפעמים אחד אחת שלושה
גדול גדולה חדש חדשה ראשון ראשונה אחרון אחרונה אותה כולם כולנו לכל מהם שהם שהיה שהייתה שבין שבו שכן כאשר כמעט בלבד מעבר בעיקר""".split())
tok=lambda s:[w for w in re.findall(r'[א-ת]{2,}',re.sub(r'[֑-ׇ]','',s))]
docs=[tok(s) for s in df.ctx]
freq=collections.Counter(w for d in docs for w in d)
PREF="והבלמשכ"
def norm(w):
    for _ in range(2):
        if len(w)>3 and w[0] in PREF and freq.get(w[1:],0)>=max(5,freq[w]): w=w[1:]
        else: break
    return w
cmap={w:norm(w) for w in freq}
KM=re.compile(r'^כמיה')
docs=[[cmap[w] for w in d] for d in docs]
docs=[[w for w in d if w not in STOP and not KM.match(w) and len(w)>1] for d in docs]
df['tokens']=[' '.join(d) for d in docs]
df=df[df.tokens.str.split().str.len()>=8]
print('контекстов после очистки:', len(df))


## 2. Выбор числа тем по когерентности (NPMI)

In [ ]:
vec = TfidfVectorizer(min_df=15, max_df=0.15, token_pattern=r"\S+", sublinear_tf=True)
X = vec.fit_transform(df.tokens); V = np.array(vec.get_feature_names_out())
B = (X > 0).astype(np.float32).tocsc(); N = B.shape[0]
def npmi(H, k=10):
    sc = []
    for t in H:
        top = np.argsort(-t)[:k]; s = []
        for i, j in itertools.combinations(top, 2):
            pa, pb = B[:, i].sum()/N, B[:, j].sum()/N; pab = B[:, i].multiply(B[:, j]).sum()/N
            s.append(np.log(pab/(pa*pb))/-np.log(pab) if pab > 0 else -1)
        sc.append(np.mean(s))
    return np.mean(sc)
for K in [8, 10, 12, 14, 16, 18]:
    print(K, round(npmi(NMF(K, init="nndsvda", random_state=0, max_iter=400).fit(X).components_), 3))

## 3. Модель с 10 темами

In [ ]:
K = 10
nmf = NMF(K, init="nndsvda", random_state=0, max_iter=400); W = nmf.fit_transform(X)
share = np.bincount(W.argmax(1), minlength=K) / len(W)
for k, t in enumerate(nmf.components_):
    print(k, f"{share[k]:.1%}", " ".join(V[np.argsort(-t)[:15]]))

## 4. Подписи тем и их сведение в 5 сфер (экспертная интерпретация по ключевым словам и 3 самым типичным контекстам каждой темы; две религиозные темы объединены, темы «место и принадлежность» и «жизнь, истина, свобода» отнесены к национальной сфере)

In [ ]:
LAB = {0:("Психология","желание, нехватка, «я»"), 1:("Нация","Земля Израиля, сионизм"), 2:("Любовь и эрос","любовь, страсть, пара"),
       3:("Религия","Храм, Иерусалим, разрушение"), 4:("Религия","человек, Бог, душа, природа"), 5:("Искусство","песни, стихи, альбомы"),
       6:("Нация","жизнь, истина, свобода"), 7:("Искусство","книги, романы, рассказы"), 8:("Нация","место и принадлежность"),
       9:("Психология","семья, дети, мужчины и женщины")}
for k in range(K):
    print(f"\n== {k}: {LAB[k]}")
    for i in np.argsort(-W[:, k])[:3]: print("   ", df.full_text.iloc[i][:200])

## 5. Сферы и их смешение
Доля сферы = доля контекстов, где её темы ведущие. Контекст «смешанный», если ≥ 25% его тематического веса приходится на каждую из двух и более сфер.

In [ ]:
M = sorted({s for s, _ in LAB.values()})
Wn = W / np.maximum(W.sum(1, keepdims=True), 1e-9)
S = np.zeros((len(W), len(M)))
for k, (s, _) in LAB.items(): S[:, M.index(s)] += Wn[:, k]
print(pd.Series(np.array(M)[S.argmax(1)]).value_counts(normalize=True).round(3))
mixed = (S >= 0.25).sum(1) >= 2
print("смешанных контекстов:", round(mixed.mean(), 3))
pairs = collections.Counter()
for row in S:
    act = [M[i] for i in np.where(row >= 0.25)[0]]
    pairs.update(itertools.combinations(sorted(act), 2))
for p, c in pairs.most_common(8): print(p, round(c/len(S), 3))